In [1]:
!pip install xgboost joblib -q

In [5]:
"""
PROJECT 1: House Price Prediction (Regression)
------------------------------------------------
Goal: Predict median house value from housing features.
Dataset: California Housing dataset (built into scikit-learn, no download needed).

Run:
    pip install -r requirements.txt
    python 01_house_price_prediction.py
"""

import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed (pip install xgboost) - skipping that model.")


MIRROR_URL = (
    "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
    "datasets/housing/housing.csv"
)


def _set_browser_user_agent():
    # Some hosts (e.g. figshare) reject requests without a normal browser
    # User-Agent header and return 403 Forbidden. This fixes that.
    opener = urllib.request.build_opener()
    opener.addheaders = [("User-Agent", "Mozilla/5.0")]
    urllib.request.install_opener(opener)


def load_data():
    _set_browser_user_agent()
    try:
        data = fetch_california_housing(as_frame=True)
        df = data.frame
        print("Downloaded via scikit-learn. Dataset shape:", df.shape)
    except Exception as e:
        print(f"scikit-learn download failed ({e}). Falling back to GitHub mirror...")
        df = pd.read_csv(MIRROR_URL)
        # Align this mirror's column names/target with the sklearn version
        df = df.rename(columns={
            "median_house_value": "MedHouseVal",
            "housing_median_age": "HouseAge",
            "total_rooms": "AveRooms",
            "total_bedrooms": "AveBedrms",
            "median_income": "MedInc",
            "population": "Population",
            "households": "AveOccup",
        })
        df = df.dropna()
        # MedHouseVal in this mirror is in raw dollars; scale to match
        # the sklearn version's units (hundreds of thousands of dollars)
        df["MedHouseVal"] = df["MedHouseVal"] / 100000
        df = df.drop(columns=["ocean_proximity", "longitude", "latitude"], errors="ignore")
        print("Downloaded via GitHub mirror. Dataset shape:", df.shape)
    print(df.head())
    return df


def explore_data(df):
    print("\n--- Summary statistics ---")
    print(df.describe())

    plt.figure(figsize=(10, 8))
    sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.savefig("correlation_heatmap.png")
    plt.close()

    plt.figure(figsize=(6, 4))
    sns.histplot(df["MedHouseVal"], kde=True)
    plt.title("Distribution of Median House Value")
    plt.tight_layout()
    plt.savefig("target_distribution.png")
    plt.close()
    print("Saved correlation_heatmap.png and target_distribution.png")


def engineer_features(df):
    df = df.copy()
    df["RoomsPerHousehold"] = df["AveRooms"] / df["AveOccup"]
    df["BedroomsPerRoom"] = df["AveBedrms"] / df["AveRooms"]
    df["PopulationPerHousehold"] = df["Population"] / df["AveOccup"]
    return df


def train_and_evaluate(X_train, X_test, y_train, y_test, model, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"\n[{name}] MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return model, {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}


def main():
    df = load_data()
    explore_data(df)
    df = engineer_features(df)

    X = df.drop(columns=["MedHouseVal"])
    y = df["MedHouseVal"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = []

    lr = LinearRegression()
    _, res = train_and_evaluate(X_train_scaled, X_test_scaled, y_train, y_test, lr, "Linear Regression")
    results.append(res)

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf, res = train_and_evaluate(X_train, X_test, y_train, y_test, rf, "Random Forest")
    results.append(res)

    if HAS_XGB:
        xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
        xgb, res = train_and_evaluate(X_train, X_test, y_train, y_test, xgb, "XGBoost")
        results.append(res)

    # Hyperparameter tuning on Random Forest as an example
    print("\nRunning GridSearchCV on Random Forest (this can take a minute)...")
    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
    }
    grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3,
                         scoring="neg_mean_squared_error", n_jobs=-1)
    grid.fit(X_train, y_train)
    print("Best params:", grid.best_params_)
    best_rf = grid.best_estimator_
    _, res = train_and_evaluate(X_train, X_test, y_train, y_test, best_rf, "Tuned Random Forest")
    results.append(res)

    # Feature importance from the best model
    importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\nFeature importances:\n", importances)
    plt.figure(figsize=(8, 5))
    importances.plot(kind="bar")
    plt.title("Feature Importance (Tuned Random Forest)")
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    plt.close()

    results_df = pd.DataFrame(results)
    print("\n--- Model comparison ---")
    print(results_df)
    results_df.to_csv("model_comparison.csv", index=False)

    joblib.dump(best_rf, "house_price_model.pkl")
    joblib.dump(scaler, "scaler.pkl")
    print("\nSaved best model to house_price_model.pkl")


if __name__ == "__main__":
    main()

scikit-learn download failed (HTTP Error 403: Forbidden). Falling back to GitHub mirror...
Downloaded via GitHub mirror. Dataset shape: (20433, 7)
   HouseAge  AveRooms  AveBedrms  Population  AveOccup  MedInc  MedHouseVal
0      41.0     880.0      129.0       322.0     126.0  8.3252        4.526
1      21.0    7099.0     1106.0      2401.0    1138.0  8.3014        3.585
2      52.0    1467.0      190.0       496.0     177.0  7.2574        3.521
3      52.0    1274.0      235.0       558.0     219.0  5.6431        3.413
4      52.0    1627.0      280.0       565.0     259.0  3.8462        3.422

--- Summary statistics ---
           HouseAge      AveRooms     AveBedrms    Population      AveOccup  \
count  20433.000000  20433.000000  20433.000000  20433.000000  20433.000000   
mean      28.633094   2636.504233    537.870553   1424.946949    499.433465   
std       12.591805   2185.269567    421.385070   1133.208490    382.299226   
min        1.000000      2.000000      1.000000      